# v8 — Data eksternal Chapman–Shaoxing (sinus takikardia asli + lebih banyak pasien AFL)

**Pertanyaan.** (1) Seberapa sering paket produksi v3.1 membunyikan alarm AFIB/AFL pada **sinus takikardia asli**?
v3.2/v7 hanya bisa mengukurnya pada data sintetis. (2) Apakah menambah ribuan pasien dari luar AFDB ke data latih
memperbaiki masalah takikardia **dan** deteksi AFL, sesuatu yang tidak bisa dicapai v3–v7 dengan 23 pasien AFDB?

**Data.** PhysioNet `ecg-arrhythmia` 1.0.0 (Chapman University, Shaoxing People's Hospital & Ningbo First Hospital,
CC BY 4.0): 45.152 pasien, satu EKG 12-lead 10 detik per pasien, 500 Hz, label ritme SNOMED CT. Hanya **lead II** yang
dipakai, diproses dengan front-end produksi (resample 50 Hz, deteksi R-peak, cek kualitas) → satu window 10 s per pasien.
Normal = SR, SB, ST, SA (irama sinus); AFIB = AFIB; AFL = AF (atrial flutter). SVT, AT, dll. tidak dipakai untuk training,
tetapi prediksinya dilaporkan.

| Varian | Data latih | Hipotesis (ditetapkan sebelum melihat hasil) |
|---|---|---|
| `v31` referensi | AFDB, resep v3/v3.1 (bobot kelas penuh) | Dilatih ulang di lingkungan ini supaya perbandingan berpasangan sah |
| `afdb` ablasi | AFDB, bobot lunak (= v5b) | Setara dengan v31 di AFDB |
| **`afdb_chapman` 8a (utama)** | AFDB + Chapman, bobot lunak | Di AFDB tidak lebih buruk dari v31; di Chapman alarm palsu sinus takikardia jauh lebih rendah dan AFL lebih baik daripada paket v3.1 |
| `chapman` 8b (eksploratif) | Chapman saja, bobot lunak | Model dari pasien lain menggeneralisasi ke AFDB? |
| `afdb_chapman_bersih` 8c (**ditambahkan setelah run pertama**) | AFDB + Chapman **tanpa label AFL** (Normal + AFIB saja) | Alarm palsu ST turun seperti 8a, AFL di AFDB tidak lebih buruk dari v31 |

**Mengapa 8c.** Run pertama menunjukkan label AFL bagian Ningbo tidak andal (bagian 2): Ningbo tidak punya satu pun AFIB,
tetapi ±7.200 "AFL"-nya berdetak ±88 bpm dan 86% tidak teratur, persis AFIB. 8a dilatih dengan label itu. 8c memakai
hanya label yang tidak diragukan. Hipotesis 8c ditulis sebelum hasilnya ada; 8a tetap dilaporkan apa adanya.

**Agar adil.** Fold AFDB resmi 5×5 (dibuktikan sama dengan fold tim), seed latih `2026 + 100·r + k`, hiperparameter
MicroTransformer sama. Pasien Chapman juga dibagi 5 fold per ulangan (diacak di dalam tiap ritme); pasien uji tidak pernah
ikut dilatih. Bias kelas cross-fit di AFDB (grid −7…+3) lalu dipakai juga di Chapman, seperti di produksi.

**Kriteria kandidat produksi untuk 8a (ditetapkan sebelum hasil).** (1) AFDB: Macro-F1 vs v31 tidak lebih buruk (P > 0,05) dan
tidak ada pasien AFL yang recall-nya turun ≥ 0,20; (2) Chapman: alarm palsu pada sinus takikardia paling banyak separuh
paket v3.1, dan recall AFL tidak lebih rendah dari paket v3.1; (3) uji stres: ambang Normal ≥ 135 bpm.

### Cara menjalankan di Kaggle
1. Notebook baru → **File → Import Notebook** → unggah file ini.
2. Panel kanan **Settings**: Accelerator **GPU T4 x2** (P100 tidak didukung PyTorch baru), **Internet: On**
   (butuh akun terverifikasi nomor HP).
3. **Save Version → Save & Run All (Commit)**. Notebook berjalan di server Kaggle (±3–4 jam); PC boleh dimatikan.
4. Setelah selesai, buka versi itu → tab **Output** → unduh `v8_hasil.zip`; unduh juga notebook-nya (berisi output).
   Bawa keduanya kembali ke repo: zip diekstrak di root repo (mengisi `cache/v8/`), notebook disalin ke folder ini.

Bisa juga dijalankan di WSL/Colab: data diunduh ke `../data/`, keluaran ke `../cache/v8/`. Job yang sudah ada dilewati.

In [ ]:
import os, sys, subprocess, shutil, time
from pathlib import Path

REPO_URL = "https://github.com/farhanlhsn/WrapSihedaf.git"
ON_KAGGLE = Path("/kaggle/working").exists()
if not Path("chapman.py").exists():                          # Kaggle/Colab: clone repo dulu
    root = Path("/kaggle/working/WrapSihedaf") if ON_KAGGLE else Path.cwd() / "WrapSihedaf"
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
    if ON_KAGGLE and not (root / "data").exists():           # data mentah (±6 GB) di disk sementara, bukan di output
        Path("/tmp/sihedaf_data").mkdir(exist_ok=True)
        (root / "data").symlink_to("/tmp/sihedaf_data")
    os.chdir(root / "v8_chapman_shaoxing")
sys.path.insert(0, os.getcwd())
try:
    import wfdb  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wfdb"], check=True)

import torch
free_gb = shutil.disk_usage(Path("../data").resolve() if Path("../data").exists() else "..").free / 1e9
print("cwd:", os.getcwd())
print("commit repo:", subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip())
print(f"Python {sys.version.split()[0]} | torch {torch.__version__} | CPU {os.cpu_count()} | disk data bebas {free_gb:.0f} GB")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA (training akan sangat lambat)")
assert free_gb > 8, "butuh ≥ 8 GB disk bebas untuk AFDB + Chapman"

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import chapman as C
cv, T = C.cv, C.T
pd.set_option("display.width", 230)
pd.set_option("display.max_columns", 30)


def fmt(v, nd=3):
    return f"{v[0]:.{nd}f} [{v[1]:.2f}–{v[2]:.2f}]"


def verdict(p):
    return "lebih baik" if p >= .95 else "lebih buruk" if p <= .05 else "tidak terbedakan"

## 1. Data AFDB dan fold resmi

Bila `../cache/dataset_v2_detector.npz` belum ada (mis. di Kaggle), AFDB diunduh (±600 MB) dan diekstrak ulang dengan ekstraktor
v5 (terbukti identik dengan v2). Hasilnya **harus** sama dengan sidik jari dataset tim (jumlah window, label, pasien, waktu);
sinyal X boleh sedikit berbeda bila versi pustaka berbeda, dan itu dilaporkan. Fold dibuat ulang dan dibuktikan sama.

In [ ]:
t0 = time.perf_counter()
info = C.ensure_afdb()
folds = C.ensure_folds()
print(f"✓ dataset AFDB cocok dengan sidik jari tim ({time.perf_counter() - t0:.0f}s):", info)
print("✓ fold resmi sama dengan fold tim:", folds.shape)
if not info["identik_bit_demi_bit"]:
    print("⚠ nilai IBI berbeda sedikit dari PC tim (versi pustaka); perbandingan di notebook ini tetap sah karena semua varian"
          " dilatih ulang di sini")

## 2. Data Chapman–Shaoxing

Data diunduh dari mirror resmi PhysioNet di AWS S3 (±5 GB, ±90 ribu file, paralel). Setiap file dicek dengan
`SHA256SUMS.txt` resmi; file yang sudah ada dan cocok dilewati, jadi unduhan bisa dilanjutkan. Ekstraksi memakai semua core
CPU dan mencetak progres setiap 2.500 pasien.

In [ ]:
t0 = time.perf_counter()
C.download_chapman()
summ = C.build_chapman_windows()
print(f"selesai {time.perf_counter() - t0:.0f}s | pasien {summ['pasien']} (unik {summ['pasien unik']}) | window valid {summ['window valid']}")
display(pd.Series(summ["alasan"], name="pasien").sort_values(ascending=False).to_frame())
display(pd.DataFrame(summ["per ritme"]).T)

### Apakah label AFL masuk akal?

AFL sungguhan (AFDB) teratur dan cepat (AFL 2:1). Bila "AFL" sebuah sumber berdetak dan tidak teratur seperti AFIB, labelnya
tidak bisa dipakai sebagai AFL. Metrik 3 kelas di Chapman (bagian 6) karena itu memakai AFL Shaoxing saja.

In [ ]:
display(pd.DataFrame(C.afl_label_check()).T.round(3))

In [ ]:
ch = C.load_chapman()
fig, ax = plt.subplots(1, 2, figsize=(13, 3.4))
for g, col, mk in (("SR", "#8A97A6", "s"), ("ST", "#2a78d6", "o"), ("AFL", "#eb6834", "^"), ("AFIB", "#1baf7a", "D")):
    m = ch["rhythm"] == g
    ax[0].hist(ch["hr"][m], bins=np.arange(30, 200, 4), histtype="step", lw=1.6, color=col, label=f"{g} ({m.sum()})", density=True)
    sel = np.flatnonzero(m)[:1500]
    ax[1].scatter(ch["hr"][sel], ch["cv"][sel], s=5, alpha=.35, color=col, marker=mk, label=g)
ax[0].set_xlabel("detak (bpm)"); ax[0].legend(frameon=False); ax[0].set_title("Sebaran detak per ritme (Chapman)", loc="left")
ax[1].set_xlabel("detak (bpm)"); ax[1].set_ylabel("CV IBI"); ax[1].set_ylim(0, .4); ax[1].legend(frameon=False)
ax[1].set_title("Keteraturan vs detak", loc="left")
plt.tight_layout(); plt.show()

## 3. Uji paritas training

Varian `v31` memakai `latih.train_dl_weighted` dengan bobot pangkat 1. Fungsi itu harus identik dengan `sihedaf_cv.train_dl`
(resep v3) pada seed yang sama.

In [ ]:
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
diff = C.parity_check(dev)
print(f"✓ paritas training (2 epoch): selisih maks {diff:.1e}" if diff < 1e-4 else f"⚠ selisih paritas {diff:.1e}")

## 4. Training (5 varian × 5 ulangan × 5 fold)

Setiap job menyimpan OOF AFDB, OOF Chapman, prediksi uji stres, dan bobot 5 model fold ke `../cache/v8/`. Job yang sudah ada
dilewati, jadi bila sesi terputus cukup jalankan ulang.

In [ ]:
C.run_all_jobs()

## 5. Hasil di AFDB (protokol v3)

Window uji detektor 50 Hz, rata-rata 5 ulangan, bias cross-fit grid −7…+3, CI 95% bootstrap pasien. Δ berpasangan terhadap
`v31` (dilatih ulang di sini) dan terhadap ablasi `afdb`.

In [ ]:
t0 = time.perf_counter()
res = C.evaluate()
print(f"evaluasi {time.perf_counter() - t0:.0f}s")
lab = res["labels"]
A = res["afdb"]
order = [k for k in ("v1", "v31", "afdb", "afdb_chapman", "chapman", "afdb_chapman_bersih") if k in A["rows"]]
display(pd.DataFrame([{"Varian": lab[k], **{m: fmt(A["rows"][k][m]) for m in ("Macro-F1", "AFL F1", "AFL R", "Normal R")},
                       "alarm AFL palsu/jam": fmt(A["episodes"][k]["alarm/jam"]),
                       "deteksi episode AFL": fmt(A["episodes"][k]["deteksi episode"], 2)} for k in order]).set_index("Varian"))
pr = []
for k in ("afdb", "afdb_chapman", "chapman", "afdb_chapman_bersih"):
    for ref, P in (("v31", A["pairs"][k]), ("afdb", A["pairs_ablasi"].get(k))):
        if P is None:
            continue
        for m in ("Macro-F1", "AFL R", "Normal R"):
            pr.append({"varian": lab[k], "vs": ref, "metrik": m, "Δ": round(P[m][0], 3), "CI 95%": f"{P[m][1]:+.3f} … {P[m][2]:+.3f}",
                       "P(Δ>0)": round(P[m][3], 3), "vonis": verdict(P[m][3])})
display(pd.DataFrame(pr).set_index(["varian", "vs", "metrik"]))
display(pd.DataFrame({lab[k]: np.round(A["afl_recall"][k], 2) for k in order}, index=A["afl_patients"]).rename_axis("recall AFL"))
display(pd.DataFrame({lab[k]: {"AFL palsu total/ulangan": round(A["fp"][k]["total"]), "dari pasien tanpa AFL": round(A["fp"][k]["pasien_lain"]),
                               "bias median": np.round(A["bias"][k]["median"], 2).tolist() if k in A["bias"] else "—"} for k in order}))

## 6. Hasil di Chapman (data eksternal)

Metrik 3 kelas pada pasien dengan ritme SR/SB/ST/SA/AFIB/AFL **berlabel andal** (AFL Ningbo dikeluarkan; lihat bagian 2).
Kolom "Macro-F1 semua label" memakai semua label, termasuk AFL Ningbo, hanya sebagai pembanding. Varian yang dilatih: OOF (pasien uji tidak ikut dilatih), bias
dari AFDB. Paket (`v31_paket`, `v2_paket`, `v1_paket`) dilatih hanya dengan AFDB, jadi seluruh Chapman adalah data uji
eksternal. CI 95% = bootstrap pasien (2.000×, bertingkat per kelas).

In [ ]:
Cc = res["chapman"]
if Cc["paket_catatan"]:
    print("⚠", Cc["paket_catatan"])
keys = [k for k in ("v1_paket", "v2_paket", "v31_paket", "v31", "afdb", "afdb_chapman", "chapman", "afdb_chapman_bersih")
        if k in Cc["metrics"]]
print("pasien untuk metrik 3 kelas (label andal):", Cc["n_metrik"])
display(pd.DataFrame([{"model": lab[k], **{m: fmt(Cc["metrics"][k][m]) for m in ("Macro-F1", "AFL F1", "AFL R", "AFIB R", "Normal R")},
                       "Macro-F1 tanpa bias": round(Cc["metrics_tanpa_bias"][k]["Macro-F1"], 3) if k in Cc["metrics_tanpa_bias"] else "—",
                       "Macro-F1 semua label": round(Cc["metrics_semua_label"][k]["Macro-F1"], 3)}
                      for k in keys]).set_index("model"))
pr = []
for ref, P in Cc["pairs"].items():
    for k in ("afdb_chapman", "chapman", "afdb", "afdb_chapman_bersih"):
        for m in ("Macro-F1", "AFL R"):
            p = P[k][m]
            pr.append({"varian": lab[k], "vs": lab[ref], "metrik": m, "Δ": round(p[0], 3), "CI 95%": f"{p[1]:+.3f} … {p[2]:+.3f}",
                       "P(Δ>0)": round(p[3], 3), "vonis": verdict(p[3])})
display(pd.DataFrame(pr).set_index(["varian", "vs", "metrik"]))

### Prediksi per ritme dan alarm palsu pada sinus takikardia asli

Angka = proporsi tebakan Normal / AFIB / AFL. Baris ST adalah pertanyaan utama v3.2: seberapa sering sinus takikardia asli
membunyikan alarm.

In [ ]:
display(pd.DataFrame([{"ritme": g, "pasien": v["n"], **{lab[k]: f"{v[k][0]:.0%} / {v[k][1]:.0%} / {v[k][2]:.0%}" for k in keys}}
                      for g, v in Cc["rhythm"].items()]).set_index("ritme"))
rows = [{"kelompok": f"ST {b} bpm", "pasien": v["n"], **{lab[k]: fmt(v[k], 2) if v["n"] else "—" for k in keys}} for b, v in Cc["st_bands"].items()]
rows.append({"kelompok": "ST semua", "pasien": Cc["st_all"]["n"], **{lab[k]: fmt(Cc["st_all"][k], 2) for k in keys}})
rows.append({"kelompok": "SR/SB/SA (sinus lain)", "pasien": Cc["sinus_alarm"]["n"], **{lab[k]: fmt(Cc["sinus_alarm"][k], 2) for k in keys}})
display(pd.DataFrame(rows).set_index("kelompok").rename_axis("alarm AFIB/AFL"))
display(pd.DataFrame([{"AFL pita": b, "pasien": v["n"], **{lab[k]: (f"{v[k]:.0%}" if v[k] is not None else "—") for k in keys}}
                      for b, v in Cc["afl_bands"].items()]).set_index("AFL pita").rename_axis("AFL terdeteksi"))

## 7. Uji stres takikardia (protokol butir 9) dan sensitivitas tanpa 04936

In [ ]:
skeys = [k for k in ("v1_paket", "v2_paket", "v31_paket", "v31", "afdb", "afdb_chapman", "chapman", "afdb_chapman_bersih")
         if k in res["threshold"]]
display(pd.DataFrame({lab[k]: res["threshold"][k] for k in skeys}).T.rename_axis("ambang (bpm) < 50% Normal"))
fig, axes = plt.subplots(1, 4, figsize=(15, 3), sharey=True)
style = {"v31_paket": ("#8A97A6", "s"), "afdb": ("#eb6834", "^"), "afdb_chapman": ("#2a78d6", "o"), "chapman": ("#1baf7a", "D"),
         "afdb_chapman_bersih": ("#172B43", "v")}
for ax, j in zip(axes, res["jitters"]):
    for k, (col, mk) in style.items():
        ax.plot(res["bpms"], [t[0] for t in res["stress"][k][j]], marker=mk, color=col, ms=3.5, label=lab[k])
    ax.set_title(j, loc="left"); ax.set_xlabel("bpm"); ax.grid(color="0.92")
axes[0].set_ylabel("ditebak Normal"); axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()
sens = []
for lab_s, s in A["sens"].items():
    for k, r in s["rows"].items():
        sens.append({"data": lab_s, "model": lab[k], **{m: fmt(r[m]) for m in ("Macro-F1", "AFL F1", "AFL R")},
                     "P(v31 > model)": round(s["pairs_v31_minus"][k][3], 2) if k in s["pairs_v31_minus"] else "—"})
display(pd.DataFrame(sens).set_index(["data", "model"]))

In [ ]:
C.CACHE_OUT.mkdir(parents=True, exist_ok=True)
(C.CACHE_OUT / "ringkasan.json").write_text(json.dumps(res, indent=1, ensure_ascii=False))
dest = Path("/kaggle/working/v8_hasil.zip") if ON_KAGGLE else C.CACHE_OUT / "v8_hasil.zip"
path, mb = C.pack_results(dest)
print(f"Ringkasan → cache/v8/ringkasan.json | hasil untuk dibawa pulang → {path} ({mb:.0f} MB)")

## 8. Kesimpulan

_(diisi setelah hasil Kaggle dibawa kembali ke repo)_